In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
anhndo_facebook_posts_transport_for_nsw_path = kagglehub.dataset_download('anhndo/facebook-posts-transport-for-nsw')

print('Data source import complete.')


# Introduction

This is an example workflow for *extracting Facebook posts and performing sentiment and other text-based analysis* on those posts. This routine can be used for any public Facebook page - and could be useful for understanding responses, sentiment of customers / audience toward company products or public issues.

In this example, I am using Transport for NSW public page - which happens to be where I work :) Here are some of the key features:
* Extracting FB posts and comments using **facebook_scraper**
* Sentiment scores using **vaderSentiment** (specialised in social media sentiment)
* Sentiment analysis for key products / services using **Spacy** library
* Use linear OLS model to investigate which posts' characteristics are strong predictors of sentiment using **statsmodels.api**

# Import Libraries

In [ ]:
# install these libraries if you don't already have them
# !pip install facebook_scraper
# !pip install vaderSentiment
# !pip install openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from collections.abc import Iterable
from facebook_scraper import get_posts
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud, STOPWORDS
import spacy
from spacy.matcher import PhraseMatcher
import statsmodels.api as sm

# Scrape posts and comments from Facebook

* Set **'comments' = True or a number (limit)** to scrape post comments
* Increase pages to capture more posts
* Doc for importing posts from public facebook page: https://github.com/kevinzg/facebook-scraper

In [ ]:
listposts = []

for post in get_posts('TransportForNSW',
                      pages=3,
                      options={'comments': True}):
    listposts.append(post)

print('Number of posts: {}'.format(len(listposts)))

# Data Processing

* Copy Raw Facebook posts data into dataframe
* Select only columns to be used for text analysis

In [ ]:
columns = ['post_id',
           'time',
           'text',
           'likes',
           'comments',
           'shares',
           'comments_full']

df_posts = pd.DataFrame(listposts)[columns]

* Create new data frame for all FB comments
* Add sentiment scores from VADER library
* Documentation for VADER sentiment scores: https://github.com/cjhutto/vaderSentiment

In [ ]:
analyzer = SentimentIntensityAnalyzer()

list_comments = []
for index, row in df_posts.iterrows():
    post_id = row['post_id']
    if isinstance(row['comments_full'], Iterable):
        for comment in row['comments_full']:
            dict_temp = {}
            dict_temp['post_id'] = post_id
            dict_temp['comment'] = comment['comment_text']
            dict_temp['sentiment'] = analyzer.polarity_scores(
                comment['comment_text'])['compound']
            list_comments.append(dict_temp)

df_comments = pd.DataFrame(list_comments)

* Calculate mean comments' sentiment scores for each posts
* Add post sentiment data to data frame
* Save all data to Excel file - for later use

In [ ]:
posts_sentiment = df_comments.groupby('post_id').mean()

df_posts = df_posts.join(posts_sentiment, on=['post_id'])

df_posts.drop(columns=['comments_full'], inplace=True)
df_posts.fillna(0.0, inplace=True)

with pd.ExcelWriter('../TfNSW_FB_new.xlsx') as writer:
    df_posts.to_excel(writer, sheet_name='posts_data', index_label='ID')
    df_comments.to_excel(writer, sheet_name='posts_comments', index_label='ID')


In [ ]:
# Load data from exisiting files to save time
df_posts = pd.read_excel('../input/facebook-posts-transport-for-nsw/TfNSW_FB.xlsx', sheet_name='posts_data')
df_comments = pd.read_excel('../input/facebook-posts-transport-for-nsw/TfNSW_FB.xlsx', sheet_name='posts_comments')

In [ ]:
# print out dataframe for posts
df_posts.head()

In [ ]:
# print out dataframe for comments
df_comments.head()

# Texts Summary - Word Cloud analysis

* Generate word cloud for all post texts

In [ ]:
# merge all texts in posts
post_text = ' '.join(df_posts['text'])
post_text = post_text.replace('\n', '') # remove blank lines characters

# update stopwords
stopwords = set(STOPWORDS)
stopwords.update(['https', 'gov', 'au', 'nsw', 's', 're'])

# Generate a word cloud image
wordcloud = WordCloud(random_state=1,
                      collocations=True,
                      stopwords=stopwords,
                      max_words=60,
                      background_color='black',
                      colormap ='rainbow',
                      contour_color='steelblue').generate(post_text)

# Function to display word cloud img via matplotlib
def plot_cloud(wordcloud):
    plt.figure(figsize=(10, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')

# plot world cloud
plot_cloud(wordcloud)

# Text Analysis with Spacy - Popularity of Products / Services
* Which TfNSw services / products are liked the most and the least?

In [ ]:
# list of common products / services
products = ['train', 'bus', 'ferry', 'light rail', 'metro',
            'road', 'bridge', 'westconnex', 'walk', 'cycling']

# Load the SpaCy model
nlp = spacy.load("en_core_web_sm")

# Create the PhraseMatcher object. The tokenizer is the first argument. Use attr = 'LOWER' to make consistent capitalization
matcher = PhraseMatcher(nlp.vocab, attr='LOWER')

# Only run nlp.make_doc to speed things up
prod_tokens = [nlp.make_doc(text) for text in products]

# Add the item patterns to the matcher
matcher.add("TfNSWProducts", prod_tokens)

## Sentiment scores and likes - based on content from FB Posts

In [ ]:
# Use defaultdict to create dictionary of sentiment scores by product
# If a key doesn't exist in item_ratings, the key is added with an empty list as the value.
product_sentiment = defaultdict(list)
product_likes = defaultdict(list)

# add sentiment scores from posts to each product
for idx, post in df_posts.iterrows():
    doc = nlp(post.loc['text'])

    # Using the matcher created before
    matches = matcher(doc)

    # Create a set of the items found in the review text
    found_products = set()
    for match_id, start, end in matches:
        product = doc[start:end].text.lower() # remove case sensitives
        found_products.add(product)

    # add sentiment scores to each products found in post
    for product in found_products:
        product_sentiment[product].append(post.sentiment)
        product_likes[product].append(post.likes)

In [ ]:
# Calculate the mean sentiment score for each product
mean_sentiment = {}

for product, score in product_sentiment.items():
    avg_score = sum(score) / len(score)
    mean_sentiment[product] = avg_score

# create dataframe for sentiment scores by product
df_prod_sentiment = pd.DataFrame(mean_sentiment, index=['sentiment']).transpose()
df_prod_sentiment = df_prod_sentiment.sort_values(by=['sentiment'], ascending=False)

# plot sentiment score by products
df_prod_sentiment.plot(kind='bar', legend=False, ylabel='Sentiment score')

* On average, posts related to 'walk' has the highest sentiment score at 0.12 (slightly possitive). Followed by 'Light rail' and 'Metro'.

In [ ]:
# Calculate the avg number of likes  for each product
mean_likes = {}

for product, likes in product_likes.items():
    avg_likes = sum(likes) / len(likes)
    mean_likes[product] = avg_likes

# create dataframe for sentiment scores by product
df_prod_likes = pd.DataFrame(mean_likes, index=['likes']).transpose()
df_prod_likes = df_prod_likes.sort_values(by=['likes'], ascending=False)

# plot sentiment score by products
df_prod_likes.plot(kind='bar', legend=False, ylabel='Number of Likes')

* On average, posts related to 'bus' have highest number of likes (~60). Followed by 'Metro' and 'Ferry'.
* Coincidentally, these modes typically have highest Customer Satisfaction ratings - per publication from TfNSW's annual surveys.

## Statistical model for Transport for NSW FB Post's Sentiment

* Using OLS model to identify which characteristics of FB Posts are good predictors for sentiment scores (for Transport NSW  public page)
* I decided to use parametric model for explainability. OLS is chosen for quick computational time.

In [ ]:
df_posts_new = df_posts # create a new df to incorporate new features

# create df for product term counts by FB posts
df_product_terms = pd.DataFrame(columns=products, index=df_posts_new.index)
df_product_terms.fillna(0, inplace=True) # remove NA in order to add increments

# count product term matches for each posts
for idx, post in df_posts_new.iterrows():
    doc = nlp(post.loc['text'])

    # Using the matcher created before
    matches = matcher(doc)

    # Create a set of the items found in the review text
    for match_id, start, end in matches:
        product = doc[start:end].text.lower() # remove case sensitives
        df_product_terms.loc[idx, product] += 1

* The below code shows routine for OLS model fit using statsmodels.api
* Based on p-values, **number of likes and comments** are strong predictors of sentiment scores, as well as number of mentions for the following modes: **light rail, road, bridge, walk**.

In [ ]:
# add product term counts to df
df_posts_new = df_posts_new.join(df_product_terms)

# predictors and target variable
features = ['likes', 'comments', 'shares'] + products
X = df_posts_new[features]
y = df_posts_new['sentiment']

# fit OLS model using statmodels api
X = sm.add_constant(X, prepend=False) # add constant term
model = sm.OLS(y, X).fit()
print(model.summary())

*There are much more we could do with rich text data from these FB posts. Please feel free to make a suggestion here or come up with your own way of analysis!*

*Thank you for reading!*